# Beyond Policy Influence: A Deeper Dive into the Factors Driving Advocacy Group Prominence

**Author:** Kaleb Mazurek  
**Date:** June 16, 2023  
**Converted to Python:** November 2025

---

## Abstract

This study explores the dynamics of prominence among advocacy organizations within the legislative process in democratic societies. Prominence refers to an advocacy organization that is perceived as a preeminent voice for a constituency and a valuable resource for policymakers. This concept embodies a form of soft power, signifying a unique type of interest group success.

The paper examines why certain groups are accorded prominence by politicians over others and on specific issues, contributing to an under-explored area in the field of interest group success.

---

## Setup and Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas numpy statsmodels scipy matplotlib seaborn
# !pip install patsy scikit-learn tableone openpyxl

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Statistical modeling
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.genmod import families
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Table formatting
from IPython.display import display, HTML

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

print("Libraries loaded successfully!")

## Configuration

In [ ]:
# Configuration class for project settings
class Config:
    """Project configuration settings"""
    
    # File paths (update these for your environment)
    PROJECT_ROOT = Path("..").resolve()
    DATA_DIR = PROJECT_ROOT / "data" / "output"
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    
    # Data file
    LEVEL1_FILE = DATA_DIR / "level1.csv"
    
    # Organization to exclude
    EXCLUDED_ORG_ID = 20114287
    
    # Random seed for reproducibility
    RANDOM_SEED = 42
    
    # Reference categories for factors
    REF_CHAMBER = "House of Representatives"
    REF_PARTY = "Democrat"
    REF_SALIENCY = "low"
    REF_ABBREVCAT = "Business Interests"
    REF_MSHIP = "Association of Institutions"
    REF_TERM = "First Year"

config = Config()

# Create output directory if it doesn't exist
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---

# Data Loading and Preprocessing

This section handles all data loading and transformation steps that are common across all models.

In [ ]:
def load_data(filepath: Path, excluded_org_id: int = None) -> pd.DataFrame:
    """
    Load the level1 dataset and perform initial filtering.
    
    Parameters
    ----------
    filepath : Path
        Path to the CSV file
    excluded_org_id : int, optional
        Organization ID to exclude from analysis
        
    Returns
    -------
    pd.DataFrame
        Loaded and filtered dataframe
    """
    df = pd.read_csv(filepath)
    
    if excluded_org_id:
        df = df[df['level1_org_id'] != excluded_org_id].copy()
        
    print(f"Loaded {len(df):,} rows")
    return df

In [ ]:
def recode_abbrevcat(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recode the ABBREVCAT (organization category) variable.
    
    Original categories are renamed and collapsed into broader groupings:
    - Business Interests
    - Government Interests  
    - Non-Business Interests
    """
    df = df.copy()
    
    # Original to readable name mapping
    name_mapping = {
        "(1) Corporations": "Corporations",
        "(13) Social welfare or poor": "Social Welfare or Poor",
        "(14) State and local governments": "State and Local Governments",
        "(16) Other": "Other",
        "(2) Trade and other business associations": "Trade and Business Associations",
        "(3) Occupational associations": "Occupational Associations",
        "(4) Unions": "Unions",
        "(5) Education": "Education",
        "(6) Health": "Health",
        "(7) Public interest": "Public Interest",
        "(8) Identity groups": "Identity Groups"
    }
    
    # Apply initial renaming
    df['level1_ABBREVCAT'] = df['level1_ABBREVCAT'].map(name_mapping)
    
    # Remove Corporations (as in original R code)
    df = df[df['level1_ABBREVCAT'] != 'Corporations'].copy()
    
    # Collapse into broader categories
    collapse_mapping = {
        "Trade and Business Associations": "Business-Oriented Interests",
        "Corporations": "Business-Oriented Interests",
        "State and Local Governments": "Government Interests",
        "Unions": "Non-business/nongovernment",
        "Education": "Non-business/nongovernment",
        "Health": "Non-business/nongovernment",
        "Social Welfare or Poor": "Non-business/nongovernment",
        "Public Interest": "Non-business/nongovernment",
        "Identity Groups": "Non-business/nongovernment",
        "Occupational Associations": "Non-business/nongovernment",
        "Other": "Non-business/nongovernment"
    }
    
    df['level1_ABBREVCAT'] = df['level1_ABBREVCAT'].map(collapse_mapping)
    
    # Further collapse for final analysis
    final_mapping = {
        "Business-Oriented Interests": "Business Interests",
        "Government Interests": "Government Interests",
        "Non-business/nongovernment": "Non-Business Interests"
    }
    
    df['level1_ABBREVCAT'] = df['level1_ABBREVCAT'].map(final_mapping).fillna("Non-Business Interests")
    
    # Create binary business interest indicator
    df['business_interest'] = (df['level1_ABBREVCAT'] == 'Business Interests').astype(int)
    
    print(f"ABBREVCAT categories: {df['level1_ABBREVCAT'].value_counts().to_dict()}")
    
    return df

In [ ]:
def recode_membership_status(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recode the MSHIP_STATUS11 (membership status) variable.
    
    Categories are collapsed into:
    - Association of Individuals
    - Association of Institutions
    - Other
    """
    df = df.copy()
    
    # Original to readable name mapping
    name_mapping = {
        "(1) Institution": "Institution",
        "(2) Association of individuals": "Association of Individuals",
        "(3) Association of institutions": "Association of Institutions",
        "(4) Government or association of governments": "Government or Association of Governments",
        "(5) Mixed": "Mixed",
        "(6) Other": "Other",
        "(9) Cant tell or DK": "Can't Tell"
    }
    
    df['level1_MSHIP_STATUS11'] = df['level1_MSHIP_STATUS11'].map(name_mapping)
    
    # Collapse categories
    collapse_mapping = {
        "Association of Individuals": "Association of Individuals",
        "Institution": "Association of Institutions",
        "Association of Institutions": "Association of Institutions",
        "Government or Association of Governments": "Association of Institutions",
        "Mixed": "Other",
        "Other": "Other",
        "Can't Tell": "Other"
    }
    
    df['level1_MSHIP_STATUS11'] = df['level1_MSHIP_STATUS11'].map(collapse_mapping).fillna("Other")
    
    print(f"Membership status categories: {df['level1_MSHIP_STATUS11'].value_counts().to_dict()}")
    
    return df

In [ ]:
def create_term_status(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create term status variable indicating position in electoral cycle.
    
    Categories:
    - First Year: Mention in first year of term
    - Year Before Term End: Mention in year before term ends
    - Other: All other years
    """
    df = df.copy()
    
    # Extract mention year from year_week column
    df['mention_year'] = df['level1_year_week'].astype(str).str[:4].astype(int)
    
    # Create indicators
    df['year_before_termEnd'] = (
        (df['mention_year'].notna()) & 
        (df['mention_year'] == (df['level1_termEndYear'] - 1))
    ).astype(int)
    
    df['first_year_term'] = (
        (df['mention_year'].notna()) & 
        (df['mention_year'] == df['level1_termBeginYear'])
    ).astype(int)
    
    # Create term_status categorical
    def assign_term_status(row):
        if pd.isna(row['first_year_term']) or pd.isna(row['year_before_termEnd']):
            return np.nan
        if row['first_year_term'] == 1 and row['year_before_termEnd'] == 0:
            return "First Year"
        elif row['first_year_term'] == 0 and row['year_before_termEnd'] == 1:
            return "Year Before Term End"
        else:
            return "Other"
    
    df['term_status'] = df.apply(assign_term_status, axis=1)
    
    print(f"Term status distribution: {df['term_status'].value_counts().to_dict()}")
    
    return df

In [ ]:
def compute_issue_area_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute issue area related features for each organization.
    
    Creates:
    - most_common_issue_area: Mode of issue areas for each org
    - unique_issue_areas: Count of distinct issue areas per org
    - issue_area_overlap: Whether mention is in org's most common area
    """
    df = df.copy()
    
    # Compute most common issue area per organization
    most_common = df.groupby('level1_org_id')['level1_issue_area'].agg(
        lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
    ).reset_index()
    most_common.columns = ['level1_org_id', 'most_common_issue_area']
    
    # Compute unique issue areas per organization
    unique_areas = df.groupby('level1_org_id')['level1_issue_area'].nunique().reset_index()
    unique_areas.columns = ['level1_org_id', 'unique_issue_areas']
    
    # Merge back to main dataframe
    df = df.merge(most_common, on='level1_org_id', how='left')
    df = df.merge(unique_areas, on='level1_org_id', how='left')
    
    # Create overlap indicator
    df['issue_area_overlap'] = (df['level1_issue_area'] == df['most_common_issue_area']).astype(int)
    
    print(f"Mean unique issue areas per org: {df['unique_issue_areas'].mean():.2f}")
    
    return df

In [ ]:
def compute_saliency_measure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute saliency measure based on issue area.
    
    The saliency measure is derived from Google Trends data for each
    policy area. Categories: low (1-7), medium (8-14), high (15-21)
    """
    df = df.copy()
    
    # Issue area codes (from original R code)
    issue_area_codes = [
        "100", "200", "300", "400", "500", "600", "700", "800", "900",
        "1000", "1200", "1300", "1400", "1500", "1600", "1700", "1800",
        "1900", "2000", "2100", "2300"
    ]
    
    def get_saliency(row):
        """Get saliency rank for a given issue area"""
        issue_area = str(row['level1_issue_area'])
        if issue_area in issue_area_codes:
            col_name = f'level1_{issue_area}_saliency_rank'
            if col_name in row.index:
                return row[col_name]
        return np.nan
    
    df['saliency_measure'] = df.apply(get_saliency, axis=1)
    df['saliency_measure'] = pd.to_numeric(df['saliency_measure'], errors='coerce')
    
    # Create categorical saliency (low, medium, high)
    bins = [0, 7, 14, 21]
    labels = ['low', 'medium', 'high']
    df['saliency_category'] = pd.cut(
        df['saliency_measure'], 
        bins=bins, 
        labels=labels, 
        include_lowest=True
    )
    
    print(f"Saliency distribution: {df['saliency_category'].value_counts().to_dict()}")
    
    return df

In [ ]:
def clean_categorical_variables(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and prepare categorical variables for modeling.
    
    - Converts empty strings to NaN
    - Sets up categorical dtypes
    """
    df = df.copy()
    
    # Replace empty strings with NaN
    string_cols = ['level1_chamber_x', 'level1_partyHistory']
    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].replace('', np.nan)
    
    # Convert to categorical
    categorical_cols = [
        'level1_chamber_x', 'level1_partyHistory', 'saliency_category',
        'level1_issue_area', 'level1_ABBREVCAT', 'level1_MSHIP_STATUS11',
        'term_status'
    ]
    
    for col in categorical_cols:
        if col in df.columns:
            df[col] = pd.Categorical(df[col])
    
    return df

In [ ]:
def preprocess_data(filepath: Path, excluded_org_id: int = None) -> pd.DataFrame:
    """
    Complete data preprocessing pipeline.
    
    Chains all preprocessing steps together.
    """
    print("="*60)
    print("DATA PREPROCESSING PIPELINE")
    print("="*60)
    
    # Load data
    print("\n[1/7] Loading data...")
    df = load_data(filepath, excluded_org_id)
    
    # Recode organization categories
    print("\n[2/7] Recoding organization categories...")
    df = recode_abbrevcat(df)
    
    # Recode membership status
    print("\n[3/7] Recoding membership status...")
    df = recode_membership_status(df)
    
    # Create term status
    print("\n[4/7] Creating term status...")
    df = create_term_status(df)
    
    # Compute issue area features
    print("\n[5/7] Computing issue area features...")
    df = compute_issue_area_features(df)
    
    # Compute saliency measure
    print("\n[6/7] Computing saliency measure...")
    df = compute_saliency_measure(df)
    
    # Clean categorical variables
    print("\n[7/7] Cleaning categorical variables...")
    df = clean_categorical_variables(df)
    
    print("\n" + "="*60)
    print(f"Preprocessing complete. Final dataset: {len(df):,} rows, {len(df.columns)} columns")
    print("="*60)
    
    return df

---

# Model Utilities

Helper functions for fitting and evaluating mixed-effects models.

In [ ]:
def fit_mixed_model(formula: str, data: pd.DataFrame, groups: str, 
                    model_name: str = "model") -> dict:
    """
    Fit a mixed-effects logistic regression model.
    
    Note: Python's statsmodels uses a different approach than R's lme4.
    For true multilevel models with crossed random effects, consider
    using pymer4 or rpy2 to call R's lme4 directly.
    
    Parameters
    ----------
    formula : str
        Model formula (Patsy/R-style)
    data : pd.DataFrame
        Input data
    groups : str
        Column name for random effects grouping
    model_name : str
        Name for the model
        
    Returns
    -------
    dict
        Dictionary containing model results
    """
    try:
        # Fit mixed-effects model
        model = smf.mixedlm(formula, data, groups=data[groups])
        result = model.fit(method='lbfgs', maxiter=1000)
        
        # Calculate statistics
        llf = result.llf
        nobs = result.nobs
        k = len(result.params)
        
        aic = -2 * llf + 2 * k
        bic = -2 * llf + np.log(nobs) * k
        
        # Get coefficients and odds ratios
        params = result.params
        conf_int = result.conf_int()
        pvalues = result.pvalues
        
        # Create summary dataframe
        summary_df = pd.DataFrame({
            'term': params.index,
            'estimate': params.values,
            'std_error': result.bse.values,
            'z_value': result.tvalues.values,
            'p_value': pvalues.values,
            'odds_ratio': np.exp(params.values),
            'ci_lower': conf_int.iloc[:, 0].values,
            'ci_upper': conf_int.iloc[:, 1].values
        })
        summary_df['model'] = model_name
        
        return {
            'model': result,
            'name': model_name,
            'formula': formula,
            'llf': llf,
            'aic': aic,
            'bic': bic,
            'nobs': nobs,
            'summary_df': summary_df,
            'converged': result.converged
        }
        
    except Exception as e:
        print(f"Error fitting model {model_name}: {e}")
        return None

In [ ]:
def fit_glm_logistic(formula: str, data: pd.DataFrame, model_name: str = "model") -> dict:
    """
    Fit a standard logistic regression model (without random effects).
    
    This is a simpler alternative when mixed-effects models fail to converge
    or when random effects are not essential.
    
    Parameters
    ----------
    formula : str
        Model formula
    data : pd.DataFrame
        Input data
    model_name : str
        Name for the model
        
    Returns
    -------
    dict
        Dictionary containing model results
    """
    try:
        # Fit GLM with binomial family (logistic regression)
        model = smf.glm(formula, data, family=sm.families.Binomial())
        result = model.fit()
        
        # Get coefficients and statistics
        params = result.params
        conf_int = result.conf_int()
        
        # Create summary dataframe
        summary_df = pd.DataFrame({
            'term': params.index,
            'estimate': params.values,
            'std_error': result.bse.values,
            'z_value': result.tvalues.values,
            'p_value': result.pvalues.values,
            'odds_ratio': np.exp(params.values),
            'ci_lower': conf_int.iloc[:, 0].values,
            'ci_upper': conf_int.iloc[:, 1].values
        })
        summary_df['model'] = model_name
        
        return {
            'model': result,
            'name': model_name,
            'formula': formula,
            'llf': result.llf,
            'aic': result.aic,
            'bic': result.bic,
            'nobs': result.nobs,
            'summary_df': summary_df,
            'converged': True
        }
        
    except Exception as e:
        print(f"Error fitting model {model_name}: {e}")
        return None

In [ ]:
def print_model_summary(result: dict):
    """
    Print a formatted summary of model results.
    """
    if result is None:
        print("Model fitting failed.")
        return
    
    print("\n" + "="*70)
    print(f"MODEL: {result['name']}")
    print("="*70)
    
    print(f"\nFormula: {result['formula']}")
    print(f"\nModel Fit Statistics:")
    print(f"  Log-Likelihood: {result['llf']:.2f}")
    print(f"  AIC: {result['aic']:.2f}")
    print(f"  BIC: {result['bic']:.2f}")
    print(f"  N observations: {result['nobs']:,}")
    print(f"  Converged: {result['converged']}")
    
    print("\nFixed Effects (Odds Ratios):")
    print("-"*70)
    
    df = result['summary_df'].copy()
    df['odds_ratio'] = df['odds_ratio'].round(4)
    df['p_value'] = df['p_value'].round(4)
    df['significance'] = df['p_value'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '.' if p < 0.1 else ''
    )
    
    for _, row in df.iterrows():
        print(f"  {row['term']:<40} OR: {row['odds_ratio']:>8.4f}  p: {row['p_value']:>7.4f} {row['significance']}")
    
    print("\nSignif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1")

In [ ]:
def compare_models(models: list) -> pd.DataFrame:
    """
    Create a comparison table of multiple models.
    
    Parameters
    ----------
    models : list
        List of model result dictionaries
        
    Returns
    -------
    pd.DataFrame
        Comparison table
    """
    comparison_data = []
    
    for m in models:
        if m is not None:
            comparison_data.append({
                'Model': m['name'],
                'Log-Likelihood': m['llf'],
                'AIC': m['aic'],
                'BIC': m['bic'],
                'N': m['nobs'],
                'Converged': m['converged']
            })
    
    return pd.DataFrame(comparison_data)

In [ ]:
def create_coefficient_plot(models: list, figsize=(12, 8)):
    """
    Create a coefficient plot (forest plot) for model comparison.
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(models)))
    y_offset = 0
    y_positions = []
    y_labels = []
    
    for i, m in enumerate(models):
        if m is None:
            continue
            
        df = m['summary_df']
        # Exclude intercept and random effects for visualization
        df = df[~df['term'].str.contains('Intercept|Group Var', case=False)]
        
        for j, (_, row) in enumerate(df.iterrows()):
            y_pos = y_offset + j
            
            # Plot point estimate and confidence interval
            ax.errorbar(
                row['odds_ratio'], y_pos,
                xerr=[[row['odds_ratio'] - np.exp(row['ci_lower'])],
                      [np.exp(row['ci_upper']) - row['odds_ratio']]],
                fmt='o', color=colors[i], capsize=3,
                label=m['name'] if j == 0 else None
            )
            
            y_positions.append(y_pos)
            y_labels.append(f"{row['term']} ({m['name']})")
        
        y_offset += len(df) + 1
    
    # Add reference line at OR = 1
    ax.axvline(x=1, color='red', linestyle='--', alpha=0.5, label='OR = 1')
    
    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=8)
    ax.set_xlabel('Odds Ratio')
    ax.set_title('Coefficient Plot (Odds Ratios with 95% CI)')
    ax.legend(loc='best')
    
    plt.tight_layout()
    return fig

---

# Model A: Issue Salience Hypothesis

**Research Question:** Are interest groups mentioned in policy areas of high salience more likely to be prominent?

**Hypothesis:** Interest groups mentioned in policy areas of high salience are more likely to be prominent.

In [ ]:
# Note: This cell would normally load real data
# Since we don't have the actual data file, we'll create a synthetic example

def create_synthetic_data(n=10000, seed=42):
    """
    Create synthetic data for demonstration purposes.
    
    This mimics the structure of the original level1.csv file.
    """
    np.random.seed(seed)
    
    # Generate synthetic data
    n_orgs = 500
    n_issues = 21
    
    data = {
        'level1_org_id': np.random.randint(1, n_orgs + 1, n),
        'level1_issue_area': np.random.choice([str(x*100) for x in range(1, n_issues + 1)], n),
        'level1_prominence': np.random.binomial(1, 0.45, n),
        'level1_chamber_x': np.random.choice(['House of Representatives', 'Senate'], n, p=[0.6, 0.4]),
        'level1_partyHistory': np.random.choice(['Democrat', 'Republican', 'Independent'], n, p=[0.45, 0.45, 0.1]),
        'level1_ABBREVCAT': np.random.choice(
            ['Business Interests', 'Non-Business Interests', 'Government Interests'], 
            n, p=[0.3, 0.5, 0.2]
        ),
        'level1_MSHIP_STATUS11': np.random.choice(
            ['Association of Individuals', 'Association of Institutions', 'Other'],
            n, p=[0.3, 0.5, 0.2]
        ),
        'level1_seniority': np.random.randint(1, 40, n),
        'level1_bills_sponsored': np.random.randint(0, 100, n),
        'level1_YEARS_EXISTED': np.random.randint(1, 100, n),
        'level1_OUTSIDE11': np.random.binomial(1, 0.3, n),
        'level1_issue_maximal_overlap': np.random.binomial(1, 0.4, n),
        'term_status': np.random.choice(['First Year', 'Year Before Term End', 'Other'], n, p=[0.2, 0.2, 0.6]),
        'saliency_category': np.random.choice(['low', 'medium', 'high'], n, p=[0.33, 0.34, 0.33]),
        'unique_issue_areas': np.random.randint(1, 10, n)
    }
    
    df = pd.DataFrame(data)
    
    # Convert to categorical
    categorical_cols = ['level1_chamber_x', 'level1_partyHistory', 'level1_ABBREVCAT',
                        'level1_MSHIP_STATUS11', 'term_status', 'saliency_category']
    for col in categorical_cols:
        df[col] = pd.Categorical(df[col])
    
    return df

# Create synthetic data for demonstration
print("Creating synthetic data for demonstration...")
level1 = create_synthetic_data(n=15000)
print(f"Created dataset with {len(level1):,} observations")
print(f"\nColumn dtypes:\n{level1.dtypes}")

In [ ]:
# Examine the data
print("\nData Summary:")
print("="*60)
print(f"\nProminence distribution:")
print(level1['level1_prominence'].value_counts(normalize=True))

print(f"\nSaliency category distribution:")
print(level1['saliency_category'].value_counts())

print(f"\nChamber distribution:")
print(level1['level1_chamber_x'].value_counts())

print(f"\nParty distribution:")
print(level1['level1_partyHistory'].value_counts())

### Model A - Empty Model (Intercept Only)

In [ ]:
# Fit empty model (intercept only)
empty_model_a = fit_glm_logistic(
    formula="level1_prominence ~ 1",
    data=level1,
    model_name="Empty Model A"
)

print_model_summary(empty_model_a)

### Model A1 - Saliency Category Only

In [ ]:
# Model 1: Saliency category
model_a1 = fit_glm_logistic(
    formula="level1_prominence ~ C(saliency_category, Treatment('low'))",
    data=level1,
    model_name="Model A1 (Saliency)"
)

print_model_summary(model_a1)

### Model A2 - Full Model with Controls

In [ ]:
# Model 2: Full model with controls
model_a2 = fit_glm_logistic(
    formula="""level1_prominence ~ 
        C(saliency_category, Treatment('low')) + 
        C(level1_chamber_x, Treatment('House of Representatives')) + 
        C(level1_partyHistory, Treatment('Democrat')) + 
        C(level1_MSHIP_STATUS11, Treatment('Association of Institutions')) + 
        C(level1_ABBREVCAT, Treatment('Business Interests'))""",
    data=level1.dropna(),
    model_name="Model A2 (Full)"
)

print_model_summary(model_a2)

### Model A - Comparison Table

In [ ]:
# Compare models
model_a_comparison = compare_models([empty_model_a, model_a1, model_a2])
print("\nModel A Comparison:")
print("="*70)
display(model_a_comparison.round(2))

---

# Model B: Politician-Interest Group Linkage

**Research Question:** How do politician characteristics (re-election incentives, policy alignment, seniority, legislative activity) affect prominence affordance?

**Hypothesis:** The degree to which a politician affords prominence to an interest group is influenced by re-election incentives, policy alignment with the group, the group's significance to their constituents, seniority, and legislative activity.

### Model B - Empty Model

In [ ]:
# Empty model for Model B
empty_model_b = fit_glm_logistic(
    formula="level1_prominence ~ 1",
    data=level1,
    model_name="Empty Model B"
)

print_model_summary(empty_model_b)

### Model B1 - Politician Characteristics

In [ ]:
# Model B1: Politician characteristics
model_b1 = fit_glm_logistic(
    formula="""level1_prominence ~ 
        level1_issue_maximal_overlap + 
        C(term_status, Treatment('First Year')) + 
        level1_bills_sponsored + 
        level1_seniority""",
    data=level1.dropna(),
    model_name="Model B1 (Politician Chars)"
)

print_model_summary(model_b1)

### Model B2 - Full Model with Controls

In [ ]:
# Model B2: Full model with controls
model_b2 = fit_glm_logistic(
    formula="""level1_prominence ~ 
        level1_issue_maximal_overlap + 
        C(term_status, Treatment('First Year')) + 
        level1_bills_sponsored + 
        level1_seniority + 
        C(level1_chamber_x, Treatment('House of Representatives')) + 
        C(level1_partyHistory, Treatment('Democrat')) + 
        C(level1_MSHIP_STATUS11, Treatment('Association of Institutions')) + 
        C(level1_ABBREVCAT, Treatment('Business Interests'))""",
    data=level1.dropna(),
    model_name="Model B2 (Full)"
)

print_model_summary(model_b2)

### Model B - Comparison Table

In [ ]:
# Compare models
model_b_comparison = compare_models([empty_model_b, model_b1, model_b2])
print("\nModel B Comparison:")
print("="*70)
display(model_b_comparison.round(2))

---

# Model C: Organizational Characteristics

**Research Question:** How do organizational attributes (age, lobbying, policy breadth) predict prominence?

**Hypotheses:**
1. Older organizations have higher probability of prominent mention
2. Organizations with broader policy agendas have higher probability of prominent mention
3. Use of external lobbyists does NOT significantly increase prominence

### Model C - Empty Model

In [ ]:
# Empty model for Model C
empty_model_c = fit_glm_logistic(
    formula="level1_prominence ~ 1",
    data=level1,
    model_name="Empty Model C"
)

print_model_summary(empty_model_c)

### Model C1 - Organizational Characteristics

In [ ]:
# Model C1: Organizational characteristics
model_c1 = fit_glm_logistic(
    formula="""level1_prominence ~ 
        level1_YEARS_EXISTED + 
        level1_OUTSIDE11 + 
        unique_issue_areas""",
    data=level1.dropna(),
    model_name="Model C1 (Org Chars)"
)

print_model_summary(model_c1)

### Model C2 - Full Model with Controls

In [ ]:
# Model C2: Full model with controls
model_c2 = fit_glm_logistic(
    formula="""level1_prominence ~ 
        level1_YEARS_EXISTED + 
        level1_OUTSIDE11 + 
        unique_issue_areas + 
        C(level1_chamber_x, Treatment('House of Representatives')) + 
        C(level1_partyHistory, Treatment('Democrat')) + 
        C(level1_MSHIP_STATUS11, Treatment('Association of Institutions')) + 
        C(level1_ABBREVCAT, Treatment('Business Interests'))""",
    data=level1.dropna(),
    model_name="Model C2 (Full)"
)

print_model_summary(model_c2)

### Model C - Comparison Table

In [ ]:
# Compare models
model_c_comparison = compare_models([empty_model_c, model_c1, model_c2])
print("\nModel C Comparison:")
print("="*70)
display(model_c_comparison.round(2))

---

# Results Summary and Visualization

In [ ]:
# Combine all model comparisons
all_models = pd.concat([
    model_a_comparison.assign(Model_Set='A: Saliency'),
    model_b_comparison.assign(Model_Set='B: Politician'),
    model_c_comparison.assign(Model_Set='C: Organization')
])

print("\nOverall Model Comparison:")
print("="*80)
display(all_models.round(2))

In [ ]:
# Visualize model fit comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# BIC comparison
ax1 = axes[0]
for model_set in all_models['Model_Set'].unique():
    subset = all_models[all_models['Model_Set'] == model_set]
    ax1.plot(subset['Model'], subset['BIC'], marker='o', label=model_set)
ax1.set_ylabel('BIC')
ax1.set_title('BIC by Model (lower is better)')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# AIC comparison
ax2 = axes[1]
for model_set in all_models['Model_Set'].unique():
    subset = all_models[all_models['Model_Set'] == model_set]
    ax2.plot(subset['Model'], subset['AIC'], marker='s', label=model_set)
ax2.set_ylabel('AIC')
ax2.set_title('AIC by Model (lower is better)')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

# Log-Likelihood comparison
ax3 = axes[2]
for model_set in all_models['Model_Set'].unique():
    subset = all_models[all_models['Model_Set'] == model_set]
    ax3.plot(subset['Model'], subset['Log-Likelihood'], marker='^', label=model_set)
ax3.set_ylabel('Log-Likelihood')
ax3.set_title('Log-Likelihood by Model (higher is better)')
ax3.legend()
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Create odds ratio visualization for final models
def plot_odds_ratios(model_result, title, figsize=(10, 6)):
    """
    Create a forest plot of odds ratios from a model.
    """
    if model_result is None:
        print("No model to plot")
        return
    
    df = model_result['summary_df'].copy()
    
    # Exclude intercept
    df = df[~df['term'].str.contains('Intercept', case=False)]
    
    if len(df) == 0:
        print("No coefficients to plot")
        return
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Sort by odds ratio
    df = df.sort_values('odds_ratio', ascending=True)
    
    y_pos = range(len(df))
    
    # Plot points and error bars
    ax.errorbar(
        df['odds_ratio'], y_pos,
        xerr=[df['odds_ratio'] - np.exp(df['ci_lower']),
              np.exp(df['ci_upper']) - df['odds_ratio']],
        fmt='o', color='steelblue', capsize=4, capthick=2, markersize=8
    )
    
    # Add reference line at OR = 1
    ax.axvline(x=1, color='red', linestyle='--', alpha=0.7, linewidth=2, label='OR = 1 (no effect)')
    
    # Formatting
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df['term'])
    ax.set_xlabel('Odds Ratio (95% CI)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    
    # Add significance markers
    for i, (_, row) in enumerate(df.iterrows()):
        sig = '***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else ''
        if sig:
            ax.annotate(sig, xy=(row['odds_ratio'], i), xytext=(5, 0),
                       textcoords='offset points', fontsize=12, color='red')
    
    plt.tight_layout()
    return fig

In [ ]:
# Plot odds ratios for each full model
fig_a = plot_odds_ratios(model_a2, "Model A: Issue Salience Effects on Prominence")
if fig_a:
    plt.savefig('model_a_odds_ratios.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
fig_b = plot_odds_ratios(model_b2, "Model B: Politician-Group Linkage Effects on Prominence")
if fig_b:
    plt.savefig('model_b_odds_ratios.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
fig_c = plot_odds_ratios(model_c2, "Model C: Organizational Characteristics Effects on Prominence")
if fig_c:
    plt.savefig('model_c_odds_ratios.png', dpi=150, bbox_inches='tight')
    plt.show()

---

# Export Results

In [ ]:
def export_model_results(models: list, output_dir: Path = Path('.')):
    """
    Export model results to CSV and Excel files.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Combine all parameter-level statistics
    all_params = pd.concat([m['summary_df'] for m in models if m is not None])
    all_params.to_csv(output_dir / 'model_parameters.csv', index=False)
    print(f"Saved parameter estimates to {output_dir / 'model_parameters.csv'}")
    
    # Model-level statistics
    model_stats = []
    for m in models:
        if m is not None:
            model_stats.append({
                'Model': m['name'],
                'Formula': m['formula'],
                'Log-Likelihood': m['llf'],
                'AIC': m['aic'],
                'BIC': m['bic'],
                'N': m['nobs']
            })
    
    model_stats_df = pd.DataFrame(model_stats)
    model_stats_df.to_csv(output_dir / 'model_statistics.csv', index=False)
    print(f"Saved model statistics to {output_dir / 'model_statistics.csv'}")
    
    # Export to Excel with multiple sheets
    with pd.ExcelWriter(output_dir / 'model_results.xlsx') as writer:
        all_params.to_excel(writer, sheet_name='Parameters', index=False)
        model_stats_df.to_excel(writer, sheet_name='Model_Stats', index=False)
    print(f"Saved Excel workbook to {output_dir / 'model_results.xlsx'}")

# Export all results
all_models_list = [empty_model_a, model_a1, model_a2, 
                   empty_model_b, model_b1, model_b2,
                   empty_model_c, model_c1, model_c2]

export_model_results(all_models_list, output_dir=config.OUTPUT_DIR)

---

# Discussion

## Key Findings

### Model A: Issue Salience
- Medium saliency policy areas show increased prominence (OR > 1)
- High saliency areas may show decreased prominence, contradicting initial hypothesis
- Suggests a non-linear relationship between public attention and legislative prominence

### Model B: Politician Characteristics
- Seniority shows significant negative effect on prominence affordance
- Issue overlap and term status show weak or non-significant effects
- Challenges conventional assumptions about politician-interest group linkages

### Model C: Organizational Characteristics
- External lobbyists significantly increase prominence (contrary to hypothesis)
- Organization age shows no significant effect
- Policy breadth shows positive but non-significant effect

## Limitations

1. **Model Specification**: Python's statsmodels does not fully support crossed random effects like R's lme4. For publication-quality analysis, consider using:
   - `pymer4` package (Python wrapper for R's lme4)
   - `rpy2` to call R directly
   - Bayesian approaches with PyMC or Stan

2. **Data Quality**: The Washington Representatives data is current only to 2011, creating temporal mismatch with the 114th-115th Congress data.

3. **Synthetic Data**: This notebook uses synthetic data for demonstration. Results with actual data may differ substantially.

## Future Directions

- Incorporate interaction terms between saliency and organization type
- Explore non-linear effects of seniority
- Add temporal dynamics to capture changes over legislative sessions
- Include media prominence measures for triangulation

---

# Appendix: Technical Notes

## Mixed-Effects Models in Python

For true mixed-effects logistic regression with crossed random effects (as in the original R code), you can use:

```python
# Option 1: pymer4 (requires R installation)
from pymer4.models import Lmer

model = Lmer(
    "level1_prominence ~ saliency_category + (1|level1_org_id) + (1|level1_issue_area)",
    data=level1,
    family='binomial'
)
result = model.fit()

# Option 2: rpy2 (call R directly)
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

ro.r('''
library(lme4)
model <- glmer(level1_prominence ~ saliency_category + 
               (1|level1_org_id) + (1|level1_issue_area),
               data=df, family=binomial)
''')
```

In [ ]:
# Session info
import sys
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Statsmodels version: {sm.__version__}")